In [2]:
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

In [4]:
# # 1. Definisi Semesta Pembicaraan (Universe)
# gpa = ctrl.Antecedent(np.arange(0, 4.1, 0.1), 'GPA')
# gre = ctrl.Antecedent(np.arange(0, 1801, 1), 'GRE')
# decision = ctrl.Consequent(np.arange(0, 101, 1), 'Decision')

# # 2. Membership Function (Sesuai Gambar 1)
# # GPA: Low (0-2.2-3.0), Medium (2.2-3.0-3.8), High (3.0-3.8-4.0)
# gpa['Low'] = fuzzy.trapmf(gpa.universe, [0, 0, 2.2, 3.0])
# gpa['Medium'] = fuzzy.trimf(gpa.universe, [2.2, 3.0, 3.8])
# gpa['High'] = fuzzy.trapmf(gpa.universe, [3.0, 3.8, 4.0, 4.0])

# # GRE: Low (0-800-1200), Medium (800-1200-1800), High (1200-1800-1800)
# gre['Low'] = fuzzy.trapmf(gre.universe, [0, 0, 800, 1200])
# gre['Medium'] = fuzzy.trimf(gre.universe, [800, 1200, 1800])
# gre['High'] = fuzzy.trapmf(gre.universe, [1200, 1800, 1800, 1800])

# # Decision (Output Sugeno seringkali menggunakan singleton, 
# # di sini kita gunakan representasi angka sesuai grafik Predikat):
# # P=60, F=70, G=80, VG=90, E=100
# decision['P'] = fuzzy.trimf(decision.universe, [0, 60, 70])
# decision['F'] = fuzzy.trimf(decision.universe, [60, 70, 80])
# decision['G'] = fuzzy.trimf(decision.universe, [70, 80, 90])
# decision['VG'] = fuzzy.trimf(decision.universe, [80, 90, 100])
# decision['E'] = fuzzy.trimf(decision.universe, [90, 100, 100])

# # 3. Rule Base (Sesuai Tabel pada Gambar)
# # Baris GPA High
# rule1 = ctrl.Rule(gpa['High'] & gre['High'], decision['E'])
# rule2 = ctrl.Rule(gpa['High'] & gre['Medium'], decision['VG'])
# rule3 = ctrl.Rule(gpa['High'] & gre['Low'], decision['F'])

# # Baris GPA Medium
# rule4 = ctrl.Rule(gpa['Medium'] & gre['High'], decision['G'])
# rule5 = ctrl.Rule(gpa['Medium'] & gre['Medium'], decision['G'])
# rule6 = ctrl.Rule(gpa['Medium'] & gre['Low'], decision['P'])

# # Baris GPA Low
# rule7 = ctrl.Rule(gpa['Low'] & gre['High'], decision['F'])
# rule8 = ctrl.Rule(gpa['Low'] & gre['Medium'], decision['P'])
# rule9 = ctrl.Rule(gpa['Low'] & gre['Low'], decision['P'])

# # 4. Simulasi
# evaluation_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
# evaluation = ctrl.ControlSystemSimulation(evaluation_ctrl)

# # Input sesuai contoh soal: GPA = 3.2 dan GRE = 900
# evaluation.input['GPA'] = 3.2
# evaluation.input['GRE'] = 900

# # Hitung (Defuzzifikasi)
# evaluation.compute()

# print(f"Hasil Evaluasi (Score): {evaluation.output['Decision']:.2f}")

# # Menentukan Predikat berdasarkan skor
# score = evaluation.output['Decision']
# if score >= 95: predikat = "Excellent (E)"
# elif score >= 85: predikat = "Very Good (VG)"
# elif score >= 75: predikat = "Good (G)"
# elif score >= 65: predikat = "Fair (F)"
# else: predikat = "Poor (P)"

# print(f"Predikat Mahasiswa: {predikat}")

# 1. Definisi Semesta Pembicaraan (Universe)
# Permintaan: 1000 - 5000
permintaan = ctrl.Antecedent(np.arange(1000, 5001, 1), 'permintaan')
# Persediaan: 100 - 600
persediaan = ctrl.Antecedent(np.arange(100, 601, 1), 'persediaan')
# Produksi: 2000 - 7000
produksi = ctrl.Consequent(np.arange(2000, 7001, 1), 'produksi')

# 2. Membership Function Linear (Sesuai Perintah Soal)
permintaan['TURUN'] = fuzzy.trimf(permintaan.universe, [1000, 1000, 5000])
permintaan['NAIK'] = fuzzy.trimf(permintaan.universe, [1000, 5000, 5000])

persediaan['SEDIKIT'] = fuzzy.trimf(persediaan.universe, [100, 100, 600])
persediaan['BANYAK'] = fuzzy.trimf(persediaan.universe, [100, 600, 600])

# 3. Mendefinisikan Output Sugeno secara Manual
# Karena skfuzzy lebih optimal untuk Mamdani, kita hitung nilai firing strength (alfa) 
# dan output secara manual agar sesuai dengan rumus di gambar.

def hitung_sugeno(input_permintaan, input_persediaan):
    # Fuzzifikasi
    mu_p_turun = fuzzy.interp_membership(permintaan.universe, permintaan['TURUN'].mf, input_permintaan)
    mu_p_naik = fuzzy.interp_membership(permintaan.universe, permintaan['NAIK'].mf, input_permintaan)
    
    mu_s_sedikit = fuzzy.interp_membership(persediaan.universe, persediaan['SEDIKIT'].mf, input_persediaan)
    mu_s_banyak = fuzzy.interp_membership(persediaan.universe, persediaan['BANYAK'].mf, input_persediaan)

    # Rule 1: IF permintaan TURUN and persediaan BANYAK THEN z1 = permintaan - persediaan
    a1 = min(mu_p_turun, mu_s_banyak)
    z1 = input_permintaan - input_persediaan

    # Rule 2: IF permintaan TURUN and persediaan SEDIKIT THEN z2 = permintaan
    a2 = min(mu_p_turun, mu_s_sedikit)
    z2 = input_permintaan

    # Rule 3: IF permintaan NAIK and persediaan BANYAK THEN z3 = permintaan
    a3 = min(mu_p_naik, mu_s_banyak)
    z3 = input_permintaan

    # Rule 4: IF permintaan NAIK and persediaan SEDIKIT THEN z4 = 1.25 * permintaan - persediaan
    a4 = min(mu_p_naik, mu_s_sedikit)
    z4 = 1.25 * input_permintaan - input_persediaan

    # Defuzzifikasi Sugeno (Weighted Average)
    total_alfa = a1 + a2 + a3 + a4
    if total_alfa == 0:
        return 0
    
    z_akhir = (a1*z1 + a2*z2 + a3*z3 + a4*z4) / total_alfa
    return z_akhir

# 4. Eksekusi sesuai soal: Permintaan 4000, Persediaan 300
hasil = hitung_sugeno(4000, 300)

print(f"--- Hasil Perhitungan Sugeno ---")
print(f"Permintaan: 4000")
print(f"Persediaan: 300")
print(f"Jumlah Produksi Barang: {hasil:.0f} kemasan")

--- Hasil Perhitungan Sugeno ---
Permintaan: 4000
Persediaan: 300
Jumlah Produksi Barang: 4230 kemasan
